# Can we predict next-month arrears?
Target: `arrears_next_month`. Plan: inspect data, check for leakage, build features known at billing time, validate with a **time-based split**, compare against simple baselines.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, precision_recall_curve
from sklearn.inspection import permutation_importance
df = pd.read_csv('data.csv', parse_dates=['billing_month']).sort_values(['meter_ref','billing_month']).reset_index(drop=True)
print(df.shape, df.isna().sum().sum(), 'missing;', df.duplicated(['meter_ref','billing_month']).sum(), 'duplicate meter-months')
print('Accounts:', df.meter_ref.nunique(), '| months:', df.billing_month.min().date(), '->', df.billing_month.max().date())
print('Arrears rate:', df.arrears_next_month.mean().round(3))
df.describe().T

## Leakage check: `final_notice_sent`

In [ ]:
print(df.groupby('final_notice_sent').arrears_next_month.agg(['mean','size']))
print('AUC of final_notice_sent alone:', round(roc_auc_score(df.arrears_next_month, df.final_notice_sent), 3))

`final_notice_sent` is "sent in the billing cycle that **followed**" the billing month, so it happens *after* the prediction point, and it nearly restates the outcome (87% arrears when 1 vs 1.4% when 0). It would not be available when making a real prediction, so **it is excluded from all models**. Including it would give a flattering but meaningless score.

## Exploration

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
df.groupby('late_payments_12m').arrears_next_month.mean().plot.bar(ax=ax[0], title='Arrears rate by late payments (12m)')
df.groupby('tariff').arrears_next_month.mean().plot.bar(ax=ax[1], title='Arrears rate by tariff')
df.groupby('billing_month').arrears_next_month.mean().plot(ax=ax[2], title='Arrears rate by month')
plt.tight_layout(); plt.show()
print(df.groupby('occupants').arrears_next_month.agg(['mean','size']))
print(df.groupby(pd.qcut(df.bill_eur, 5), observed=True).arrears_next_month.mean())

## Drift: a regime change in Jan 2025
Average bill jumps ~15% from Jan 2025 (about 112 to 128 EUR) while average kWh and late-payment counts stay flat, and the arrears rate climbs from ~3% to ~26% by Dec 2025. So the base rate in the test period is far above anything in training, which matters for calibration and for how far historical patterns can be trusted.

In [ ]:
m_ = df.groupby('billing_month').agg(arrears=('arrears_next_month','mean'), bill=('bill_eur','mean'), kwh=('kwh','mean'), late=('late_payments_12m','mean'))
fig, ax = plt.subplots(1, 3, figsize=(15, 3.3))
m_.arrears.plot(ax=ax[0], title='Arrears rate'); m_.bill.plot(ax=ax[1], title='Mean bill (EUR)'); m_.kwh.plot(ax=ax[2], title='Mean kWh')
plt.tight_layout(); plt.show()

The panel grows over time (36 accounts in Jul 2023, 1,375 in Dec 2025), so early months are small and noisy.

## Features
Only information available at billing time: current-row fields, ratios, and **lagged** per-account history (previous rows only).

In [ ]:
g = df.groupby('meter_ref')
df['price_per_kwh'] = df.bill_eur / df.kwh
df['kwh_per_occ'] = df.kwh / df.occupants
df['bill_prev'] = g.bill_eur.shift(1)
df['bill_change'] = df.bill_eur / df.bill_prev - 1
df['late_prev'] = g.late_payments_12m.shift(1)
df['late_delta'] = df.late_payments_12m - df.late_prev
df['kwh_vs_own_mean'] = df.kwh / g.kwh.transform(lambda s: s.shift(1).expanding().mean())
df['n_obs'] = g.cumcount()
df['month'] = df.billing_month.dt.month
num = ['occupants','kwh','bill_eur','late_payments_12m','price_per_kwh','kwh_per_occ','bill_change','late_delta','kwh_vs_own_mean','n_obs','month']
X = pd.concat([df[num], pd.get_dummies(df['tariff'], prefix='tariff', dtype=float)], axis=1)
X = X.replace([np.inf, -np.inf], np.nan)
y = df.arrears_next_month
assert 'final_notice_sent' not in X.columns
X.shape

## Validation
Time-based split: train on billing months up to **2025-05**, skip June (a one-month gap, since the target looks a month ahead), test on **2025-07 to 2025-12**. This mimics deploying the model forward in time. The same accounts appear in both periods, as they would in real deployment.

In [ ]:
tr = (df.billing_month <= '2025-05-01').values
te = (df.billing_month >= '2025-07-01').values
print('train', tr.sum(), y[tr].mean().round(3), '| test', te.sum(), y[te].mean().round(3))
fill = X[tr].median()
Xf = X.fillna(fill)
models = {
  'Base rate only': None,
  'late_payments only': LogisticRegression(),
  'Logistic regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
  'Gradient boosting': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, l2_regularization=1.0, random_state=0),
}
rows, preds = [], {}
for n, m in models.items():
    if m is None:
        p = np.full(te.sum(), y[tr].mean())
    else:
        cols = ['late_payments_12m'] if n == 'late_payments only' else list(X.columns)
        m.fit(Xf.loc[tr, cols], y[tr]); p = m.predict_proba(Xf.loc[te, cols])[:, 1]
    preds[n] = p
    rows.append((n, roc_auc_score(y[te], p), average_precision_score(y[te], p), brier_score_loss(y[te], p)))
res = pd.DataFrame(rows, columns=['model','ROC AUC','PR AUC','Brier']).set_index('model').round(3)
res

In [ ]:
print('Test base rate (PR AUC of a random model):', y[te].mean().round(3))
best = 'Gradient boosting'
p = preds[best]
prec, rec, thr = precision_recall_curve(y[te], p)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(rec, prec); ax[0].axhline(y[te].mean(), ls='--', c='grey'); ax[0].set(xlabel='recall', ylabel='precision', title='Precision-recall (test)')
q = pd.qcut(p, 10, duplicates='drop')
cal = pd.DataFrame({'pred': p, 'obs': y[te].values}).groupby(q, observed=True).mean()
ax[1].plot(cal.pred, cal.obs, 'o-'); ax[1].plot([0,.8],[0,.8],'--',c='grey'); ax[1].set(xlabel='mean predicted', ylabel='observed rate', title='Calibration by decile (test)')
plt.tight_layout(); plt.show()

In [ ]:
# What does flagging the top-k% riskiest accounts capture?
t = pd.DataFrame({'p': p, 'y': y[te].values}).sort_values('p', ascending=False)
out = []
for k in (.05,.10,.20,.30):
    s = t.head(int(len(t)*k)); out.append((f'top {int(k*100)}%', s.y.mean(), s.y.sum()/t.y.sum(), s.y.mean()/t.y.mean()))
pd.DataFrame(out, columns=['flagged','precision','recall','lift']).round(3)

## Which features matter?

In [ ]:
m = models[best]
pi = permutation_importance(m, Xf[te], y[te], scoring='roc_auc', n_repeats=5, random_state=0)
pd.Series(pi.importances_mean, X.columns).sort_values().tail(10).plot.barh(title='Permutation importance (AUC drop, test)'); plt.show()

## Robustness: rolling-origin check
Repeat train/test over several cut-offs (3-month test windows, 1-month gap) to see whether the result depends on one split.

In [ ]:
rows = []
for cut in pd.date_range('2025-01-01','2025-09-01',freq='2MS'):
    a = (df.billing_month < cut - pd.offsets.MonthBegin(1)).values
    b = ((df.billing_month >= cut) & (df.billing_month < cut + pd.offsets.MonthBegin(3))).values
    mdl = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, l2_regularization=1.0, random_state=0).fit(X[a], y[a])
    rows.append((cut.date(), a.sum(), b.sum(), roc_auc_score(y[b], mdl.predict_proba(X[b])[:,1]), roc_auc_score(y[b], df.late_payments_12m[b])))
pd.DataFrame(rows, columns=['test_from','n_train','n_test','GB AUC','late_payments AUC']).round(3)

## Effect of the drift: recency and calibration
Models trained on the low-arrears era under-predict risk in the test period (see the calibration plot). Check whether training only on the new regime (Jan to May 2025) helps ranking, and how well-calibrated a simple recalibration would be.

In [ ]:
tr2 = ((df.billing_month >= '2025-01-01') & (df.billing_month <= '2025-05-01')).values
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
rows = []
for name, mask in [('train all history', tr), ('train 2025-01..05 only', tr2)]:
    lr.fit(Xf[mask], y[mask]); pp = lr.predict_proba(Xf[te])[:,1]
    rows.append((name, mask.sum(), y[mask].mean(), pp.mean(), y[te].mean(), roc_auc_score(y[te], pp), average_precision_score(y[te], pp), brier_score_loss(y[te], pp)))
pd.DataFrame(rows, columns=['logistic trained on','n_train','train rate','mean pred (test)','test rate','ROC AUC','PR AUC','Brier']).round(3)

## Conclusion
**Answer: partly.** Next-month arrears can be ranked better than chance from information known at billing time, but not strongly, and the model is not reliable as a calibrated probability.

- **Ranking skill (time-based test, Jul-Dec 2025):** ROC AUC about 0.72-0.75 for the best models vs 0.70 for `late_payments_12m` alone. PR AUC is about 0.47-0.52 vs a 0.22 base rate. Flagging the top 10% of accounts catches roughly 26% of arrears cases at about 58% precision (lift about 2.7).
- **Most of the signal is payment history** (`late_payments_12m`) plus bill size (arrears rate rises from 5% in the lowest bill quintile to 22% in the highest). The extra features add only a few AUC points, and logistic regression is as good as or better than gradient boosting, so use the simpler model.
- **Rolling-origin check:** the gain over the late-payments baseline is inconsistent. Early windows show the baseline ahead, later ones show the model ahead, so treat the improvement as modest.
- **Drift is the main risk.** From Jan 2025 bills rise about 15% and the arrears rate climbs from about 3% to 26%. Models trained on pre-2025 data underestimate risk (Brier is barely better than the base rate), and training only on the recent regime improves ranking a bit but uses little data. Probabilities need recalibration on recent months before being used for decisions, and the model should be monitored and retrained.
- **`final_notice_sent` was excluded on purpose.** It is sent *after* the prediction point and nearly restates the outcome (AUC 0.94 on its own). Using it would look impressive but be unusable in practice.

**Caveats:** a single time-ordered holdout of six months; only 1,375 accounts, and the same accounts appear in train and test; the cause of the Jan 2025 shift (price rise? policy change?) is unknown and should be confirmed with the business; and the right alert threshold depends on the cost of a missed arrears case vs a false alarm, which is not specified here.